In [3]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.6 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0
    Uninstalling pip-26.0:
      Successfully uninstalled pip-26.0


In [4]:
!pip -q install tensorflow scikit-learn pandas numpy matplotlib seaborn

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


In [5]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEED = 34
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TF version:', tf.__version__)
print('All imports OK')

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
DATA_DIR = Path('.')

SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x']     = out['accel_x'].clip(-128, 127)
    out['accel_y']     = out['accel_y'].clip(-128, 127)
    out['accel_z']     = out['accel_z'].clip(-128, 127)
    out['eda']         = out['eda'].clip(0, 60)
    out['heart_rate']  = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id']        = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid']       = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress']    = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA  = clean_sensor(pd.read_csv(DATA_DIR / 'train-sensor.csv'))
TRAIN_LABEL = clean_label(pd.read_csv(DATA_DIR / 'train-label.csv'))
TEST_DATA   = clean_sensor(pd.read_csv(DATA_DIR / 'test-sensor.csv'))
TEST_LABEL  = clean_label(pd.read_csv(DATA_DIR / 'test-label.csv'))

print('TRAIN_DATA :', TRAIN_DATA.shape)
print('TRAIN_LABEL:', TRAIN_LABEL.shape)
print('TEST_DATA  :', TEST_DATA.shape)
print('TEST_LABEL :', TEST_LABEL.shape)

In [ ]:
train = pd.merge_asof(
    TRAIN_DATA.sort_values('timestamp'),
    TRAIN_LABEL[['pid', 'timestamp', 'stress']].sort_values('timestamp'),
    on='timestamp', by='pid', direction='nearest'
)
train = train.dropna(subset=['stress']).reset_index(drop=True)
train['stress'] = train['stress'].astype(int)   # must be int for sparse_categorical

print('Merged shape:', train.shape)
print('Label distribution:')
print(train['stress'].value_counts().sort_index())

In [ ]:
FEATURE_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

data = train[FEATURE_COLS + ['stress']].copy()

# per-column StandardScaler — store each scaler for reuse on test
scalers = {}
for col in FEATURE_COLS:
    sc = StandardScaler()
    data[col] = sc.fit_transform(np.array(data[col]).reshape(len(data[col]), 1))
    scalers[col] = sc

print('Scaled sample:')
print(data.head(3))

In [ ]:
Feature = data[FEATURE_COLS]
Target  = data['stress']

f_train, f_test, t_train, t_test = train_test_split(
    Feature, Target, test_size=0.4, random_state=SEED
)
print('f_train:', f_train.shape)
print('f_test :', f_test.shape)

In [ ]:
callback = EarlyStopping(
    monitor='val_loss',
    min_delta=0.0001,
    patience=10,
    verbose=1,
    mode='auto',
    restore_best_weights=True   # fixed: None is not valid in newer Keras
)

model = Sequential([
    Dense(32, activation='tanh', input_dim=6),
    Dense(10, activation='tanh'),
    Dense(3,  activation='softmax'),
])

model.summary()

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    f_train, t_train,
    validation_split=0.2,
    epochs=100,
    verbose=1,
    batch_size=800,
    callbacks=[callback]
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     color='green', label='Training')
plt.plot(history.history['val_accuracy'], color='red',   label='Validation')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     color='green', label='Training')
plt.plot(history.history['val_loss'], color='red',   label='Validation')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
loss, accuracy = model.evaluate(f_test, t_test, verbose=0)
print(f'Test Loss    : {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

In [ ]:
HALF_WIN = 160   # 160 samples each side → 320 total window @ 32Hz

def extract_window_mean(sensor_df, pid, ts_ms):
    grp = sensor_df[sensor_df['pid'] == pid].reset_index(drop=True)
    if grp.empty:
        return None
    idx   = (grp['timestamp'] - ts_ms).abs().idxmin()
    start = max(0, idx - HALF_WIN)
    end   = start + 320
    if end > len(grp):
        end   = len(grp)
        start = max(0, end - 320)
    w = grp.iloc[start:end]
    return w if len(w) > 0 else None

test_rows = []
test_ids  = []

for _, row in TEST_LABEL.iterrows():
    pid = str(row['pid'])
    ts  = row['timestamp']
    rid = int(row['id'])

    w = extract_window_mean(TEST_DATA, pid, ts)
    if w is None:
        test_rows.append({c: 0.0 for c in SENSOR_COLS})
    else:
        test_rows.append({c: w[c].mean() for c in SENSOR_COLS})
    test_ids.append(rid)

X_test_raw = pd.DataFrame(test_rows, columns=SENSOR_COLS)

# rename to match FEATURE_COLS order used in training
X_test_raw = X_test_raw[FEATURE_COLS]

# apply same per-column scalers fitted on train
for col in FEATURE_COLS:
    X_test_raw[col] = scalers[col].transform(
        np.array(X_test_raw[col]).reshape(len(X_test_raw[col]), 1)
    )

print('Test feature shape:', X_test_raw.shape)

# predict
y_prob = model.predict(X_test_raw, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

submission = pd.DataFrame({'id': test_ids, 'stress': y_pred.astype(int)})
submission.to_csv('subn3.csv', index=False)

print(f'Saved subn1.csv — {len(submission)} rows')
print('Prediction distribution:')
print(submission['stress'].value_counts().sort_index())
print(submission.head())